# Barcode Detection with Dynamsoft DBR

This notebook tests Dynamsoft Barcode Reader as an atomic decoder. It reads the 300 DPI rendered page cache directly, sends each page to DBR, converts returned locations into boxes, and writes the visual review output.


In [ ]:
from pathlib import Path
import json
import subprocess
import shutil
import math
import os

from dynamsoft_barcode_reader_bundle import *
from PIL import Image, ImageDraw, ImageOps

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

def find_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'data' / 'pdfs').exists():
            return path
    raise RuntimeError('Could not find repo root containing data/pdfs')


ROOT = find_root()
PDF_PATH = ROOT / 'data' / 'pdfs' / 'private-evaluation-document.pdf'
PAGES = list(range(1, 22))

OUT_DIR = ROOT / 'output' / 'dbr-atom'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300
GRID_COLS = 2
CELL_WIDTH = 600
DBR_TEMPLATE = EnumPresetTemplate.PT_READ_BARCODES.value
DBR_LICENSE_FALLBACK = 
DBR_LICENSE = os.getenv('DYNAMSOFT_LICENSE_KEY', DBR_LICENSE_FALLBACK)

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'barcode-research'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')



In [ ]:
def init_dbr():
    if not DBR_LICENSE:
        raise RuntimeError('Dynamsoft needs a license. Set DYNAMSOFT_LICENSE_KEY.')
    error_code, error_message = LicenseManager.init_license(DBR_LICENSE)
    if error_code not in (0, 1):
        raise RuntimeError(f'Dynamsoft license error {error_code}: {error_message}')
    return CaptureVisionRouter()


DBR_ROUTER = None


def get_dbr_router():
    global DBR_ROUTER
    if DBR_ROUTER is None:
        DBR_ROUTER = init_dbr()
    return DBR_ROUTER


def box_from_item(item):
    return [[float(point.x), float(point.y)] for point in item.get_location().points]


def read_barcodes(image_path):
    result = get_dbr_router().capture(str(image_path), DBR_TEMPLATE)
    if result.get_error_code() != 0:
        return []
    decoded = result.get_decoded_barcodes_result()
    if decoded is None:
        return []
    detections = []
    for item in decoded.get_items() or []:
        box = box_from_item(item)
        detections.append({
            'format': item.get_format_string(),
            'raw_value': item.get_text(),
            'confidence': float(item.get_confidence() or 0),
            'box': box,
        })
    return detections


def draw_detections(image_path, detections):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow']
    for i, det in enumerate(detections, start=1):
        points = [tuple(point) for point in det['box']]
        color = colors[(i - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=6)
        x = min(p[0] for p in points)
        y = max(0, min(p[1] for p in points) - 26)
        label = f'{i} {det["format"]}: {det["raw_value"][:28]}'
        draw.rectangle([x, y, x + min(560, max(220, 8 * len(label))), y + 26], fill=color)
        draw.text((x + 4, y + 5), label, fill='black')
    return image


def create_grid(images, cols=2, cell_width=600):
    thumbs = []
    for image in images:
        ratio = cell_width / image.width
        thumbs.append(image.resize((cell_width, int(image.height * ratio))))
    rows = math.ceil(len(thumbs) / cols)
    cell_height = max(image.height for image in thumbs)
    grid = Image.new('RGB', (cols * cell_width, rows * cell_height), 'white')
    for i, image in enumerate(thumbs):
        grid.paste(image, ((i % cols) * cell_width, (i // cols) * cell_height))
    return grid


Run the DBR baseline over all pages and save JSON plus preview images.


In [ ]:
render_pdf_if_needed()
OUT_DIR.mkdir(parents=True, exist_ok=True)

results = []
preview_images = []
total_found = 0

for page in PAGES:
    image_path = PAGE_DIR / f'page-{page:02d}.png'
    if not image_path.exists():
        print(f'Skipping page {page}: image not found')
        continue

    detections = read_barcodes(image_path)
    total_found += len(detections)
    preview = draw_detections(image_path, detections)
    preview_path = OUT_DIR / f'page-{page:02d}.png'
    preview.save(preview_path)
    preview_images.append(preview)

    print(f'page {page:02d}: found {len(detections)}')
    results.append({
        'page': page,
        'count': len(detections),
        'detections': detections,
        'debug_image': str(preview_path),
    })

(OUT_DIR / 'detections.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'\nTotal: {total_found} barcodes across {len(results)} pages')
print(f'Saved JSON: {OUT_DIR / "detections.json"}')

if preview_images:
    grid = create_grid(preview_images, cols=GRID_COLS, cell_width=CELL_WIDTH)
    grid_path = OUT_DIR / 'all-pages-grid.png'
    grid.save(grid_path)
    print(f'Saved grid: {grid_path}')
    display(grid)
else:
    print('No pages processed.')

